In [3]:
!pip install pythainlp

   ---------------------------------------- 0.0/19.3 MB ? eta -:--:--
   ---------------- ----------------------- 7.9/19.3 MB 44.1 MB/s eta 0:00:01
   ---------------------------------------  18.9/19.3 MB 52.3 MB/s eta 0:00:01
   ---------------------------------------  18.9/19.3 MB 52.3 MB/s eta 0:00:01
   ---------------------------------------- 19.3/19.3 MB 29.4 MB/s  0:00:00



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import csv
import os
import glob
import re

from pythainlp.tokenize import word_tokenize

# ==== CONFIG ====
INPUT_CSV = r"D:\Documents\University\Senior Project\gender-bias-detection\services\scraper\src\scraper\blog_based\tweets_data.csv"

OUTPUT_DIR = "chunks"                  # Output folder
ROWS_PER_FILE = 100                   # Rows per chunk (เฉพาะ data ไม่รวม header)
OUTPUT_PREFIX = "tweets_part_"         # Prefix for chunk files

TEXT_COLUMN = "text"                   # Column with original Thai text
TOKENIZED_COLUMN = "tokens"            # Column for tokenized text
# =================


def preprocess_text(text: str) -> str:
    if not text:
        return ""

    # แทน newline, tab → space
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")

    # ทำอังกฤษให้เป็น lower-case (normalize case)
    text = text.lower()

    # ตัด whitespace ซ้ำ (รวม space หลายตัว ให้เหลือ 1 ตัว)
    text = re.sub(r"\s+", " ", text).strip()

    # ยุบ ... ยาวๆ
    text = re.sub(r"\.{3,}", "...", text)
    text = re.sub(r"…+", "…", text)

    # -----------------------------
    # 🔥 Normalize "laughing" → "หัวเราะ"
    # -----------------------------

    # 555+, 55555+, 555555, 555555++++
    text = re.sub(r"5{3,}\+*", "หัวเราะ", text)

    # ฮ่า, ฮ่าๆ, ฮ่าๆๆๆๆๆๆๆ
    text = re.sub(r"(ฮ่า)+", "หัวเราะ", text)

    # ฮา, ฮาๆๆๆ
    text = re.sub(r"(ฮา)+", "หัวเราะ", text)

    # ภาษาอังกฤษ haha, hahaha, HAHAHA, haha+, hahaha!!!
    # (?i) = case-insensitive แต่เรา lower ไปแล้วก็โอเคเหมือนกัน
    text = re.sub(r"(?i)ha(ha)+\+*", "หัวเราะ", text)

    # หลังการแทน อาจเกิดหัวเราะหัวเราะ → ยุบเหลือหัวเราะเดียว
    text = re.sub(r"(หัวเราะ)+", "หัวเราะ", text)

    # -----------------------------
    # ตัวเลขยาวๆ 5 หลักขึ้นไป (คุณตั้งเป็น "หัวเราะ" ไว้ ถ้าอยากเปลี่ยนเป็น <num> ก็คือบรรทัดนี้)
    # -----------------------------
    text = re.sub(r"\d{5,}", "หัวเราะ", text)

    # -----------------------------
    # ลบ URL, mention, hashtag (ถ้าไม่อยากลบ เอา 3 บรรทัดนี้ออกได้)
    # -----------------------------
    text = re.sub(r"http\S+|www\.\S+", " ", text)          # URL
    text = re.sub(r"@[A-Za-z0-9_]+", " ", text)           # @username
    text = re.sub(r"#[A-Za-z0-9_]+", " ", text)           # #hashtag

    # ยุบ !!!, ???, ?!?!? → ! หรือ ? เดียว
    text = re.sub(r"([!?])\1{1,}", r"\1", text)           # !!! -> !
    text = re.sub(r"([!?]){2,}", r"\1", text)             # ?!?! -> ?

    # เก็บเฉพาะชุดอักษรที่ต้องการ
    text = re.sub(r"[^0-9A-Za-z\u0E00-\u0E7F\s\.\,\!\?\']", " ", text)

    # Final compress whitespace อีกครั้งกันหลุด
    text = re.sub(r"\s+", " ", text).strip()

    return text



def split_csv_with_thai_tokens(
    input_csv: str,
    output_dir: str,
    rows_per_file: int = 100,
    output_prefix: str = "part_",
    text_column: str = "text",
    tokenized_column: str = "tokens",
):
    os.makedirs(output_dir, exist_ok=True)

    # ลบไฟล์เก่าก่อน เพื่อไม่ให้ข้อมูลเก่าปน
    pattern = os.path.join(output_dir, f"{output_prefix}*.csv")
    for old_file in glob.glob(pattern):
        os.remove(old_file)

    with open(input_csv, "r", newline="", encoding="utf-8") as f_in:
        reader = csv.DictReader(f_in)

        if reader.fieldnames is None:
            print("Input CSV has no header or is empty.")
            return

        fieldnames = list(reader.fieldnames)
        if tokenized_column not in fieldnames:
            fieldnames.append(tokenized_column)

        file_index = 0
        row_count = 0
        writer = None
        out_file = None
        current_file_path = None

        def open_new_file(idx: int):
            nonlocal current_file_path
            filename = f"{output_prefix}{idx:04d}.csv"
            path = os.path.join(output_dir, filename)
            current_file_path = path
            f_out = open(path, "w", newline="", encoding="utf-8")
            w = csv.DictWriter(f_out, fieldnames=fieldnames)
            w.writeheader()
            print(f"Created {path}")
            return f_out, w

        for row in reader:
            # ---- PREPROCESS ----
            raw_text = row.get(text_column) or ""
            clean_text = preprocess_text(raw_text)

            # ---- TOKENIZE (ไทย + อังกฤษปนกัน) ----
            # PyThaiNLP จะตัดคำไทย + แยกอังกฤษตามช่องว่างไปด้วย
            tokens = word_tokenize(clean_text, engine="newmm") if clean_text else []

            # เก็บเป็น string แบบ list เช่น "['พี่', 'นัท', '555']"
            row[tokenized_column] = repr(tokens)

            # ---- SPLIT ----
            if writer is None or row_count >= rows_per_file:
                if out_file is not None:
                    out_file.close()
                    print(f" -> {current_file_path} has {row_count} data rows")

                file_index += 1
                out_file, writer = open_new_file(file_index)
                row_count = 0

            writer.writerow(row)
            row_count += 1

        if out_file is not None:
            out_file.close()
            print(f" -> {current_file_path} has {row_count} data rows")

    print(
        f"Done. Split into {file_index} file(s) in '{output_dir}' "
        f"with up to {rows_per_file} rows each (last may be smaller)."
    )


if __name__ == "__main__":
    split_csv_with_thai_tokens(
        input_csv=INPUT_CSV,
        output_dir=OUTPUT_DIR,
        rows_per_file=ROWS_PER_FILE,
        output_prefix=OUTPUT_PREFIX,
        text_column=TEXT_COLUMN,
        tokenized_column=TOKENIZED_COLUMN,
    )


Created chunks\tweets_part_0001.csv
 -> chunks\tweets_part_0001.csv has 100 data rows
Created chunks\tweets_part_0002.csv
 -> chunks\tweets_part_0002.csv has 100 data rows
Created chunks\tweets_part_0003.csv
 -> chunks\tweets_part_0003.csv has 100 data rows
Created chunks\tweets_part_0004.csv
 -> chunks\tweets_part_0004.csv has 100 data rows
Created chunks\tweets_part_0005.csv
 -> chunks\tweets_part_0005.csv has 100 data rows
Created chunks\tweets_part_0006.csv
 -> chunks\tweets_part_0006.csv has 100 data rows
Created chunks\tweets_part_0007.csv
 -> chunks\tweets_part_0007.csv has 100 data rows
Created chunks\tweets_part_0008.csv
 -> chunks\tweets_part_0008.csv has 100 data rows
Created chunks\tweets_part_0009.csv
 -> chunks\tweets_part_0009.csv has 100 data rows
Created chunks\tweets_part_0010.csv
 -> chunks\tweets_part_0010.csv has 100 data rows
Created chunks\tweets_part_0011.csv
 -> chunks\tweets_part_0011.csv has 100 data rows
Created chunks\tweets_part_0012.csv
 -> chunks\tweets_